In [29]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

from statsmodels.tsa.stattools import adfuller
from arch.unitroot import PhillipsPerron

from statsmodels.tsa.ardl import ardl_select_order

In [30]:
# ============================================================
# Load processed data
# ============================================================
data_processed = pd.read_csv('../data/processed/data_processed.csv')

# ============================================================
# Max lag
# ============================================================
max_lag = 6

# ============================================================
# Cấu hình 3 model song song
# ============================================================
models = {
    "Model_1_TB": {
        "dep_var": "ln_TB",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff",
                      "ln_M2_diff","ln_WTI","COVID"]
    },
    "Model_2_EX": {
        "dep_var": "ln_EX",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff",
                      "ln_M2_diff","ln_WTI","COVID"]
    },
    "Model_3_IM": {
        "dep_var": "ln_IM",
        "exog_vars": ["ln_RER_pos","ln_RER_neg","ln_FDI","IPI_VN","IPI_World_GR_diff",
                      "ln_M2_diff","ln_WTI","COVID"]
    }
}

# ============================================================
# Chạy Auto-ARDL để chọn lag tối ưu
# ============================================================
selected_models = {}
results_summary = []

for model_name, spec in models.items():
    dep_var = spec["dep_var"]
    exog_vars = [v for v in spec["exog_vars"] if v in data_processed.columns]

    # Lấy dataframe cho model, bỏ NA
    df_model = data_processed[[dep_var] + exog_vars].dropna()

    # Auto ARDL lag selection
    sel = ardl_select_order(
        endog=df_model[dep_var],
        exog=df_model[exog_vars],
        maxlag=max_lag,
        maxorder=max_lag,
        ic='aic',
        trend='c'
    )

    # Fit ARDL model
    fitted_model_results = sel.model.fit()

    # Lưu fitted model
    selected_models[model_name] = fitted_model_results

    # Lưu summary vào list để export
    summary_df = pd.read_html(fitted_model_results.summary().tables[1].as_html(), header=0, index_col=0)[0]
    summary_df['Model'] = model_name
    results_summary.append(summary_df)

    print("\n" + "="*80)
    print(f"Optimal Lag Selection for {model_name}")
    print("="*80)
    print(fitted_model_results.summary())

# ============================================================
# Xuất kết quả summary ra folder results
# ============================================================
output_folder = "../results"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "02_optimal_lag_selection.csv")

# Gộp tất cả model vào 1 file
pd.concat(results_summary).to_csv(output_file)
print(f"\nĐã xuất ARDL/NARDL lag summary ra: {output_file}")


Optimal Lag Selection for Model_1_TB
                               ARDL Model Results                              
Dep. Variable:                   ln_TB   No. Observations:                  131
Model:             ARDL(5, 0, 6, 0, 6)   Log Likelihood                 208.944
Method:                Conditional MLE   S.D. of innovations              0.046
Date:                 Mon, 25 May 2026   AIC                           -371.887
Time:                         02:23:03   BIC                           -306.653
Sample:                              6   HQIC                          -345.384
                                   131                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0493      0.040      1.240      0.218      -0.030       0.128
ln_TB.L1          0.2157      0.092      2.350      0.021       0.034       